# Chapter 11: Correlation and Regression

Source span: printed pages 245-266; PDF pages 260-281. The source was inspected for orientation only: section order, terminology, and the kinds of statistics used. This notebook uses original prose, synthetic data, generated diagrams, and executable checks.

## Chapter Goal

Dependence between directional variables is not a single number copied from linear statistics. The sample space decides what a residual means, what transformations should be ignored, and which null model is fair. By the end of this chapter notebook you should be able to:

- measure a linear-circular relation by embedding the circle as sine-cosine coordinates;
- compare two circular variables on the torus, including positive and negative angular dependence;
- fit a rotation-based spherical regression and read tangent residuals on the sphere;
- use permutation nulls to decide whether a directional statistic is large for the geometry rather than large by accident;
- recognize serial dependence in a circular time series without creating artificial jumps at the angle cut.

## Computational Translation Guide

| Textbook object | Computational representation | Invariance to check |
| --- | --- | --- |
| Linear variable paired with an angle | regress `x` on `cos(theta)` and `sin(theta)` | shifting the angular origin rotates the two predictors but leaves the multiple correlation unchanged |
| Two circular variables | points `(theta, phi)` on a torus; angular residuals use wrapped differences | separate choices of zero direction for both circles should not change dependence strength |
| Spherical-spherical pair | unit rows `X, Y` in `R^3`; covariance blocks or product matrices | separate rotations of the two coordinate frames should not change vector correlation |
| Spherical regression | estimate an element of `SO(3)` by Procrustes/Kabsch matching | fitted directions remain unit length; residual vectors are tangent to the sphere |
| Directional time series | lagged pairs `(theta_t, theta_{t+1})`; statistics from wrapped increments | persistence should survive the angle cut and exceed a permutation null |

The route through the notebook is visualization-first. Each statistic is paired with an artifact and a check: a cylinder for line-circle dependence, a torus for circle-circle residuals, a sphere for rotation regression, and a lag plot for time-series dependence.

In [ ]:
from pathlib import Path
import sys


def find_book_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (
            (candidate / "AGENTS.md").exists()
            and (candidate / "scripts" / "validate_dirstats_course.py").exists()
            and (candidate / "utils").exists()
        ):
            return candidate
    raise RuntimeError("Could not locate Directional-Statistics course root")


BOOK_ROOT = find_book_root(Path.cwd())
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

TOPIC = "chapter-11"
ARTIFACT_BASE = BOOK_ROOT / "artifacts" / TOPIC
source_span = {"printed_pages": "245-266", "pdf_pages": "260-281"}
source_span

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
from scipy import stats
from scipy.spatial.transform import Rotation

from utils.artifacts import display_artifact, save_json, save_matplotlib, save_plotly_html
from utils.validation import assert_artifacts

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(20261111)


def wrap_pi(angle):
    """Wrap radians to [-pi, pi)."""
    return (np.asarray(angle) + np.pi) % (2 * np.pi) - np.pi


def unit_rows(array):
    array = np.asarray(array, dtype=float)
    norms = np.linalg.norm(array, axis=1, keepdims=True)
    if np.any(norms == 0):
        raise ValueError("zero vector cannot be normalized")
    return array / norms


def random_unit_sphere(n, generator):
    return unit_rows(generator.normal(size=(n, 3)))


def perturb_on_sphere(points, scale, generator):
    noise = generator.normal(size=points.shape)
    tangent = noise - np.sum(noise * points, axis=1, keepdims=True) * points
    return unit_rows(points + scale * tangent)


def circle_embed(theta):
    return np.column_stack([np.cos(theta), np.sin(theta)])


def linear_circular_r2(x, theta):
    design = np.column_stack([np.ones_like(theta), np.cos(theta), np.sin(theta)])
    beta, *_ = np.linalg.lstsq(design, x, rcond=None)
    fitted = design @ beta
    sse = float(np.sum((x - fitted) ** 2))
    sst = float(np.sum((x - np.mean(x)) ** 2))
    return 1.0 - sse / sst, beta, fitted


def linear_circular_r2_formula(x, theta):
    c = np.cos(theta)
    s = np.sin(theta)
    corr = np.corrcoef(np.column_stack([x, c, s]), rowvar=False)
    r_xc, r_xs, r_cs = corr[0, 1], corr[0, 2], corr[1, 2]
    return float((r_xc**2 + r_xs**2 - 2 * r_xc * r_xs * r_cs) / (1 - r_cs**2))


def vector_correlation_r2(x, y):
    x_centered = x - np.mean(x, axis=0, keepdims=True)
    y_centered = y - np.mean(y, axis=0, keepdims=True)
    n = x.shape[0]
    s_xx = x_centered.T @ x_centered / n
    s_yy = y_centered.T @ y_centered / n
    s_xy = x_centered.T @ y_centered / n
    return float(np.trace(np.linalg.pinv(s_xx) @ s_xy @ np.linalg.pinv(s_yy) @ s_xy.T))


def circular_dependence(theta, phi):
    positive = abs(np.mean(np.exp(1j * (phi - theta)))) ** 2
    negative = abs(np.mean(np.exp(1j * (phi + theta)))) ** 2
    offset = float(np.angle(np.mean(np.exp(1j * (phi - theta)))))
    residual = wrap_pi(phi - theta - offset)
    embedded_r2 = vector_correlation_r2(circle_embed(theta), circle_embed(phi))
    return {
        "positive_rotation_strength": float(positive),
        "negative_rotation_strength": float(negative),
        "signed_strength": float(positive - negative),
        "offset_radians": offset,
        "angular_residuals": residual,
        "embedded_r2": float(embedded_r2),
    }


def fit_so3_rotation_rows(x, y):
    """Return R with x @ R approximately y and det(R)=1."""
    h = x.T @ y
    u, singular_values, vt = np.linalg.svd(h)
    correction = np.eye(3)
    correction[-1, -1] = np.linalg.det(u @ vt)
    r = u @ correction @ vt
    return r, singular_values


def artifact_is_under(path, root):
    try:
        path.resolve().relative_to(root.resolve())
        return True
    except ValueError:
        return False


def quantile_summary(null_values, observed):
    null_values = np.asarray(null_values, dtype=float)
    return {
        "observed": float(observed),
        "null_mean": float(np.mean(null_values)),
        "null_q95": float(np.quantile(null_values, 0.95)),
        "permutation_p_value": float((1 + np.sum(null_values >= observed)) / (len(null_values) + 1)),
        "observed_gt_null_q95": bool(observed > np.quantile(null_values, 0.95)),
    }

## 1. Linear-Circular Dependence: A Cylinder, Not A Line

For a linear response `x` and an angle `theta`, the chapter's embedding idea turns `theta` into `(cos(theta), sin(theta))`. The multiple correlation from regressing `x` on those two columns answers a geometric question: does height on the cylinder vary systematically with direction around the cylinder?

Inspect the red ridge in the artifact. It is a fitted sinusoid drawn on the cylinder wall. The histogram beside it is a permutation null: the same heights are randomly reassigned to angles, so any large statistic must come from the pairing, not from the marginal spread of `x` or the marginal spread of angles.

In [ ]:
n_lc = 84
theta_lc = rng.uniform(-np.pi, np.pi, n_lc)
x_lc = 12.0 + 3.2 * np.cos(theta_lc - 0.85) + rng.normal(0, 0.75, n_lc)

lc_r2, lc_beta, lc_fitted = linear_circular_r2(x_lc, theta_lc)
lc_r2_formula = linear_circular_r2_formula(x_lc, theta_lc)
lc_null = np.array([linear_circular_r2(x_lc, rng.permutation(theta_lc))[0] for _ in range(500)])
shift = 1.37
lc_shifted_r2 = linear_circular_r2(x_lc, wrap_pi(theta_lc + shift))[0]

grid = np.linspace(-np.pi, np.pi, 240)
grid_design = np.column_stack([np.ones_like(grid), np.cos(grid), np.sin(grid)])
grid_fit = grid_design @ lc_beta
z_min, z_max = float(x_lc.min() - 0.8), float(x_lc.max() + 0.8)

fig = plt.figure(figsize=(12, 4.4))
ax = fig.add_subplot(1, 2, 1, projection="3d")
mesh_theta, mesh_z = np.meshgrid(np.linspace(-np.pi, np.pi, 72), np.linspace(z_min, z_max, 2))
ax.plot_surface(np.cos(mesh_theta), np.sin(mesh_theta), mesh_z, color="#b9c4cf", alpha=0.12, linewidth=0)
scatter = ax.scatter(np.cos(theta_lc), np.sin(theta_lc), x_lc, c=x_lc, cmap="viridis", s=38, depthshade=False)
ax.plot(1.04 * np.cos(grid), 1.04 * np.sin(grid), grid_fit, color="#d62728", linewidth=2.5, label="fitted ridge")
ax.set_title("Line-circle sample on a cylinder")
ax.set_xlabel("cos(theta)")
ax.set_ylabel("sin(theta)")
ax.set_zlabel("x")
ax.set_box_aspect((1, 1, 0.8))
ax.view_init(elev=20, azim=-54)

ax2 = fig.add_subplot(1, 2, 2)
ax2.hist(lc_null, bins=28, color="#ced8e4", edgecolor="white")
ax2.axvline(lc_r2, color="#d62728", linewidth=2.5, label="observed")
ax2.axvline(np.quantile(lc_null, 0.95), color="#384860", linestyle="--", linewidth=1.8, label="null 95%")
ax2.set_title("Permutation null for line-circle R^2")
ax2.set_xlabel("R^2 after shuffling angles")
ax2.set_ylabel("count")
ax2.legend(frameon=False)
fig.colorbar(scatter, ax=ax, shrink=0.72, pad=0.08, label="x")
fig.tight_layout()

linear_png = save_matplotlib(fig, TOPIC, "core", "linear-circular-cylinder-null.png")
plt.close(fig)
display_artifact(linear_png, width=940)

linear_checks = {
    "r2_lstsq": float(lc_r2),
    "r2_formula": float(lc_r2_formula),
    "formula_vs_lstsq_abs_error": float(abs(lc_r2 - lc_r2_formula)),
    "origin_shifted_r2": float(lc_shifted_r2),
    "origin_invariance_abs_error": float(abs(lc_r2 - lc_shifted_r2)),
    "null_comparison": quantile_summary(lc_null, lc_r2),
}
linear_checks

## 2. Circular-Circular Dependence: The Torus And Angular Residuals

Two angles live on `S^1 x S^1`, a torus. A diagonal band on the torus represents rotational dependence: `phi` is approximately `theta + offset`. The residual is not `phi - theta - offset` as an ordinary real number; it is the wrapped angle left after the fitted offset is removed.

The artifact uses three linked views. The torus view shows the paired observations. The polar residual view shows whether the fitted offset leaves a concentrated cloud around zero. The null histogram shows how often the same positive-rotation strength appears after the `phi` values are permuted.

In [ ]:
n_cc = 96
theta_cc = rng.uniform(-np.pi, np.pi, n_cc)
phi_cc = wrap_pi(theta_cc + 0.92 + rng.vonmises(0.0, 8.0, n_cc))
cc = circular_dependence(theta_cc, phi_cc)
cc_null = np.array([
    circular_dependence(theta_cc, rng.permutation(phi_cc))["positive_rotation_strength"]
    for _ in range(500)
])
cc_embedded_p = float(stats.chi2.sf(n_cc * cc["embedded_r2"], df=4))
cc_shifted = circular_dependence(wrap_pi(theta_cc - 0.61), wrap_pi(phi_cc + 1.42))

major, minor = 2.0, 0.58
torus_u, torus_v = np.meshgrid(np.linspace(-np.pi, np.pi, 70), np.linspace(-np.pi, np.pi, 34))
torus_x = (major + minor * np.cos(torus_v)) * np.cos(torus_u)
torus_y = (major + minor * np.cos(torus_v)) * np.sin(torus_u)
torus_z = minor * np.sin(torus_v)
point_x = (major + minor * np.cos(phi_cc)) * np.cos(theta_cc)
point_y = (major + minor * np.cos(phi_cc)) * np.sin(theta_cc)
point_z = minor * np.sin(phi_cc)
band_theta = np.linspace(-np.pi, np.pi, 240)
band_phi = wrap_pi(band_theta + cc["offset_radians"])
band_x = (major + 1.04 * minor * np.cos(band_phi)) * np.cos(band_theta)
band_y = (major + 1.04 * minor * np.cos(band_phi)) * np.sin(band_theta)
band_z = 1.04 * minor * np.sin(band_phi)
residual = cc["angular_residuals"]

fig = plt.figure(figsize=(15, 4.4))
ax = fig.add_subplot(1, 3, 1, projection="3d")
ax.plot_surface(torus_x, torus_y, torus_z, color="#d7dde5", alpha=0.18, linewidth=0)
pts = ax.scatter(point_x, point_y, point_z, c=np.abs(residual), cmap="magma", s=32, depthshade=False)
ax.plot(band_x, band_y, band_z, color="#1f77b4", linewidth=2.2)
ax.set_title("Pairs as points on the torus")
ax.set_axis_off()
ax.set_box_aspect((1, 1, 0.45))
ax.view_init(elev=23, azim=-38)

ax2 = fig.add_subplot(1, 3, 2, projection="polar")
ax2.scatter(residual, np.ones_like(residual), c=np.abs(residual), cmap="magma", s=28)
ax2.plot([0, 0], [0, 1.14], color="#1f77b4", linewidth=2)
ax2.set_title("Wrapped angular residuals")
ax2.set_yticklabels([])
ax2.set_rlim(0, 1.18)

ax3 = fig.add_subplot(1, 3, 3)
ax3.hist(cc_null, bins=28, color="#d9dfdf", edgecolor="white")
ax3.axvline(cc["positive_rotation_strength"], color="#d62728", linewidth=2.4, label="observed")
ax3.axvline(np.quantile(cc_null, 0.95), color="#384860", linestyle="--", linewidth=1.8, label="null 95%")
ax3.set_title("Permutation null for rotation strength")
ax3.set_xlabel("|mean exp(i(phi - theta))|^2")
ax3.set_ylabel("count")
ax3.legend(frameon=False)
fig.colorbar(pts, ax=ax, shrink=0.62, pad=0.02, label="|residual|")
fig.tight_layout()

circular_png = save_matplotlib(fig, TOPIC, "core", "circular-circular-torus-residual-null.png")
plt.close(fig)
display_artifact(circular_png, width=980)

circular_checks = {
    "positive_rotation_strength": float(cc["positive_rotation_strength"]),
    "negative_rotation_strength": float(cc["negative_rotation_strength"]),
    "signed_strength": float(cc["signed_strength"]),
    "embedded_r2": float(cc["embedded_r2"]),
    "embedded_chi2_reference_p_value": cc_embedded_p,
    "separate_origin_invariance_abs_error": float(abs(cc["positive_rotation_strength"] - cc_shifted["positive_rotation_strength"])),
    "max_abs_residual_le_pi": bool(np.max(np.abs(residual)) <= np.pi),
    "mean_abs_residual_radians": float(np.mean(np.abs(residual))),
    "null_comparison": quantile_summary(cc_null, cc["positive_rotation_strength"]),
}
circular_checks

## 3. Spherical-Spherical Dependence And Regression By Rotation

For paired unit vectors `x_i, y_i` on `S^2`, an embedded correlation uses covariance blocks of the two vector samples. It is invariant under separate rotations of the coordinate frames. Regression asks a different question: is there a rotation `R` such that `x_i @ R` predicts `y_i`?

The fitted rotation is found by an SVD of the cross-product matrix, with a determinant correction that keeps the estimate in `SO(3)`. Residuals are tangent vectors at the fitted points: the radial component is removed, so the residual lives in the local tangent plane rather than inside the ball.

In [ ]:
n_sp = 110
x_sp = random_unit_sphere(n_sp, rng)
true_rotation = Rotation.from_euler("zyx", [36, -22, 18], degrees=True)
y_clean = true_rotation.apply(x_sp)
y_sp = perturb_on_sphere(y_clean, 0.10, rng)

r_row, singular_values = fit_so3_rotation_rows(x_sp, y_sp)
y_hat = unit_rows(x_sp @ r_row)
sp_r2 = vector_correlation_r2(x_sp, y_sp)
sp_null = np.array([vector_correlation_r2(x_sp, rng.permutation(y_sp)) for _ in range(400)])
sp_align = float(np.mean(np.sum(y_hat * y_sp, axis=1)))
sp_align_null_values = []
for _ in range(250):
    permuted = rng.permutation(y_sp)
    r_perm, _ = fit_so3_rotation_rows(x_sp, permuted)
    sp_align_null_values.append(np.mean(np.sum((x_sp @ r_perm) * permuted, axis=1)))
sp_align_null = np.array(sp_align_null_values)

rot_x = Rotation.from_euler("xyz", [12, 31, -9], degrees=True).as_matrix().T
rot_y = Rotation.from_euler("zyx", [-28, 7, 41], degrees=True).as_matrix().T
sp_rotated_r2 = vector_correlation_r2(x_sp @ rot_x, y_sp @ rot_y)
residual_vectors = y_sp - np.sum(y_sp * y_hat, axis=1, keepdims=True) * y_hat
angular_errors = np.arccos(np.clip(np.sum(y_sp * y_hat, axis=1), -1.0, 1.0))
sp_chi2_p = float(stats.chi2.sf(n_sp * sp_r2, df=9))

sphere_u, sphere_v = np.meshgrid(np.linspace(0, 2 * np.pi, 70), np.linspace(0, np.pi, 35))
sphere_x = np.cos(sphere_u) * np.sin(sphere_v)
sphere_y = np.sin(sphere_u) * np.sin(sphere_v)
sphere_z = np.cos(sphere_v)
subset = np.linspace(0, n_sp - 1, 22, dtype=int)

fig = plt.figure(figsize=(13.5, 4.5))
ax = fig.add_subplot(1, 2, 1, projection="3d")
ax.plot_surface(sphere_x, sphere_y, sphere_z, color="#d5dde8", alpha=0.16, linewidth=0)
ax.scatter(y_sp[:, 0], y_sp[:, 1], y_sp[:, 2], c=angular_errors, cmap="plasma", s=32, depthshade=False, label="observed y")
ax.scatter(y_hat[:, 0], y_hat[:, 1], y_hat[:, 2], color="#1f77b4", s=12, alpha=0.65, depthshade=False, label="fitted R x")
for idx in subset:
    ax.plot(
        [y_hat[idx, 0], y_sp[idx, 0]],
        [y_hat[idx, 1], y_sp[idx, 1]],
        [y_hat[idx, 2], y_sp[idx, 2]],
        color="#323b4a",
        alpha=0.55,
        linewidth=0.8,
    )
ax.set_title("Rotation regression on S^2")
ax.set_axis_off()
ax.set_box_aspect((1, 1, 1))
ax.view_init(elev=18, azim=-48)

ax2 = fig.add_subplot(1, 2, 2)
ax2.hist(sp_null, bins=26, color="#d9e0ea", edgecolor="white", label="permutation null")
ax2.axvline(sp_r2, color="#d62728", linewidth=2.4, label="observed vector r^2")
ax2.axvline(np.quantile(sp_null, 0.95), color="#384860", linestyle="--", linewidth=1.8, label="null 95%")
ax2.set_title("Spherical-spherical vector correlation")
ax2.set_xlabel("trace(Sxx^-1 Sxy Syy^-1 Syx)")
ax2.set_ylabel("count")
ax2.legend(frameon=False)
fig.tight_layout()

spherical_png = save_matplotlib(fig, TOPIC, "core", "spherical-spherical-procrustes-residuals.png")
plt.close(fig)
display_artifact(spherical_png, width=940)

# Interactive artifact: rotate the fitted sphere scene and inspect residual size.
fig3d = go.Figure()
fig3d.add_trace(go.Surface(x=sphere_x, y=sphere_y, z=sphere_z, colorscale=[[0, "#edf1f6"], [1, "#edf1f6"]], opacity=0.22, showscale=False, name="unit sphere"))
fig3d.add_trace(go.Scatter3d(
    x=y_sp[:, 0], y=y_sp[:, 1], z=y_sp[:, 2], mode="markers",
    marker=dict(size=4, color=angular_errors, colorscale="Plasma", colorbar=dict(title="angle error")),
    text=[f"angular error {err:.3f} rad" for err in angular_errors],
    name="observed y",
))
fig3d.add_trace(go.Scatter3d(
    x=y_hat[:, 0], y=y_hat[:, 1], z=y_hat[:, 2], mode="markers",
    marker=dict(size=3, color="#1f77b4"), name="fitted R x",
))
for idx in subset:
    fig3d.add_trace(go.Scatter3d(
        x=[y_hat[idx, 0], y_sp[idx, 0]],
        y=[y_hat[idx, 1], y_sp[idx, 1]],
        z=[y_hat[idx, 2], y_sp[idx, 2]],
        mode="lines",
        line=dict(color="#303846", width=2),
        showlegend=False,
    ))
axis_vec = Rotation.from_matrix(r_row.T).as_rotvec()
axis_norm = np.linalg.norm(axis_vec)
if axis_norm > 0:
    axis_vec = axis_vec / axis_norm
    fig3d.add_trace(go.Scatter3d(
        x=[-axis_vec[0], axis_vec[0]],
        y=[-axis_vec[1], axis_vec[1]],
        z=[-axis_vec[2], axis_vec[2]],
        mode="lines",
        line=dict(color="#d62728", width=6),
        name="estimated rotation axis",
    ))
fig3d.update_layout(
    title="Spherical regression: fitted rotation, observed directions, tangent-size residuals",
    scene=dict(aspectmode="data", xaxis_title="x", yaxis_title="y", zaxis_title="z"),
    margin=dict(l=0, r=0, t=42, b=0),
    legend=dict(orientation="h", y=0.02),
)
spherical_html = save_plotly_html(fig3d, TOPIC, "interactive", "spherical-rotation-regression.html", include_plotlyjs=True)
display_artifact(spherical_html, width="100%", height=560)

spherical_checks = {
    "vector_correlation_r2": float(sp_r2),
    "embedded_chi2_reference_p_value": sp_chi2_p,
    "separate_rotation_invariance_abs_error": float(abs(sp_r2 - sp_rotated_r2)),
    "fit_rotation_det": float(np.linalg.det(r_row)),
    "fit_rotation_orthogonality_error": float(np.linalg.norm(r_row.T @ r_row - np.eye(3))),
    "prediction_unit_norm_max_error": float(np.max(np.abs(np.linalg.norm(y_hat, axis=1) - 1.0))),
    "residual_tangent_max_abs_dot": float(np.max(np.abs(np.sum(residual_vectors * y_hat, axis=1)))),
    "mean_angular_residual_radians": float(np.mean(angular_errors)),
    "alignment_null_comparison": quantile_summary(sp_align_null, sp_align),
    "vector_correlation_null_comparison": quantile_summary(sp_null, sp_r2),
}
spherical_checks

## 4. Directional Time-Series Intuition

A time series of angles should not be analyzed by pretending that `pi` and `-pi` are far apart. The lag statistic below uses adjacent wrapped increments. A persistent series has small increments, so the mean resultant of `exp(i(theta_{t+1} - theta_t))` is large.

The artifact shows three diagnostics: the path around the circle through time, the wrapped angle series, and a permutation null obtained by shuffling the same angles before computing adjacent increments. Shuffling preserves the marginal distribution of directions and breaks serial order.

In [ ]:
n_ts = 170
mu_ts = 2.88
rho_ts = 0.88
sigma_ts = 0.30
theta_ts = np.empty(n_ts)
theta_ts[0] = wrap_pi(mu_ts + rng.normal(0, 0.8))
for t in range(1, n_ts):
    innovation = rng.normal(0, sigma_ts)
    theta_ts[t] = wrap_pi(mu_ts + rho_ts * wrap_pi(theta_ts[t - 1] - mu_ts) + innovation)

lag_increment = wrap_pi(theta_ts[1:] - theta_ts[:-1])
lag_resultant = float(abs(np.mean(np.exp(1j * lag_increment))))
lag_cosine_mean = float(np.mean(np.cos(lag_increment)))
ts_null = np.array([
    abs(np.mean(np.exp(1j * wrap_pi(np.diff(rng.permutation(theta_ts))))))
    for _ in range(500)
])

fig = plt.figure(figsize=(14.5, 4.2))
ax = fig.add_subplot(1, 3, 1)
points = circle_embed(theta_ts)
ax.plot(points[:, 0], points[:, 1], color="#69788f", linewidth=1.2, alpha=0.75)
ax.scatter(points[:, 0], points[:, 1], c=np.arange(n_ts), cmap="viridis", s=22)
unit = np.linspace(0, 2 * np.pi, 240)
ax.plot(np.cos(unit), np.sin(unit), color="#222222", linewidth=0.8)
ax.set_title("Ordered path on the circle")
ax.set_aspect("equal")
ax.set_xticks([])
ax.set_yticks([])

ax2 = fig.add_subplot(1, 3, 2)
ax2.scatter(np.arange(n_ts), theta_ts, c=np.arange(n_ts), cmap="viridis", s=18)
ax2.set_title("Angles stay wrapped")
ax2.set_xlabel("time")
ax2.set_ylabel("theta_t in [-pi, pi)")
ax2.set_ylim(-np.pi - 0.2, np.pi + 0.2)

ax3 = fig.add_subplot(1, 3, 3)
ax3.hist(ts_null, bins=28, color="#dde3dd", edgecolor="white")
ax3.axvline(lag_resultant, color="#d62728", linewidth=2.4, label="observed")
ax3.axvline(np.quantile(ts_null, 0.95), color="#384860", linestyle="--", linewidth=1.8, label="null 95%")
ax3.set_title("Permutation null for lag resultant")
ax3.set_xlabel("|mean exp(i Delta theta)|")
ax3.set_ylabel("count")
ax3.legend(frameon=False)
fig.tight_layout()

time_png = save_matplotlib(fig, TOPIC, "core", "directional-time-series-lag-null.png")
plt.close(fig)
display_artifact(time_png, width=980)

time_checks = {
    "lag_resultant": lag_resultant,
    "lag_cosine_mean": lag_cosine_mean,
    "increments_wrapped_to_pi": bool(np.max(np.abs(lag_increment)) <= np.pi),
    "null_comparison": quantile_summary(ts_null, lag_resultant),
}
time_checks

## Applied Lab: Noise Weakens Circular Dependence

This small lab varies only the concentration of angular noise around the model `phi = theta + offset + noise`. The target is not a formal model fit; it is a calibration exercise. Watch the positive-rotation statistic, the embedded vector correlation, and the mean absolute angular residual as concentration changes. Strong concentration gives a narrow torus band. Low concentration spreads points around the torus and pushes the observed statistics toward their permutation null behavior.

In [ ]:
lab_theta = rng.uniform(-np.pi, np.pi, 160)
lab_offset = 0.7
lab_rows = []
for kappa in [0.35, 0.75, 1.5, 3.5, 9.0, 18.0]:
    lab_phi = wrap_pi(lab_theta + lab_offset + rng.vonmises(0.0, kappa, lab_theta.size))
    lab_summary = circular_dependence(lab_theta, lab_phi)
    lab_rows.append({
        "noise_kappa": float(kappa),
        "positive_rotation_strength": float(lab_summary["positive_rotation_strength"]),
        "embedded_r2": float(lab_summary["embedded_r2"]),
        "mean_abs_residual_radians": float(np.mean(np.abs(lab_summary["angular_residuals"]))),
    })

sweep_path = save_json({"rows": lab_rows}, TOPIC, "checks", "circular-noise-sweep.json")
lab_rows

## Consolidated Checks

The last code cells collect the chapter-specific invariants. They are deliberately redundant with the visuals: every major artifact has a nearby geometric claim and a final machine-checkable version of that claim.

Checks included:

- linear-circular multiple correlation matches the explicit correlation formula and is unchanged by shifting the angular origin;
- circular-circular positive-rotation strength is unchanged by separate origin shifts and angular residuals stay wrapped;
- spherical vector correlation is unchanged by separate rotations, the fitted regression matrix lies in `SO(3)`, predictions are unit vectors, and residuals are tangent;
- time-series lag dependence exceeds a permutation null while all increments are wrapped;
- all generated artifacts are inside the chapter-11 artifact subtree and have nonzero size.

In [ ]:
chapter_checks = {
    "source_span": source_span,
    "linear_circular": linear_checks,
    "circular_circular": {k: v for k, v in circular_checks.items() if k != "angular_residuals"},
    "spherical_spherical": spherical_checks,
    "directional_time_series": time_checks,
    "lab": {"noise_sweep_rows": len(lab_rows)},
}
checks_path = save_json(chapter_checks, TOPIC, "checks", "correlation-regression-invariants.json")
chapter_checks

In [ ]:
artifact_paths = [
    linear_png,
    circular_png,
    spherical_png,
    spherical_html,
    time_png,
    sweep_path,
    checks_path,
]
artifact_records = assert_artifacts(artifact_paths, min_bytes=100)
assert all(artifact_is_under(path, ARTIFACT_BASE) for path in artifact_paths), artifact_records

assert linear_checks["formula_vs_lstsq_abs_error"] < 1e-12
assert linear_checks["origin_invariance_abs_error"] < 1e-12
assert linear_checks["null_comparison"]["observed_gt_null_q95"]

assert circular_checks["separate_origin_invariance_abs_error"] < 1e-12
assert circular_checks["max_abs_residual_le_pi"]
assert circular_checks["null_comparison"]["observed_gt_null_q95"]

assert spherical_checks["separate_rotation_invariance_abs_error"] < 1e-10
assert abs(spherical_checks["fit_rotation_det"] - 1.0) < 1e-10
assert spherical_checks["fit_rotation_orthogonality_error"] < 1e-10
assert spherical_checks["prediction_unit_norm_max_error"] < 1e-12
assert spherical_checks["residual_tangent_max_abs_dot"] < 1e-12
assert spherical_checks["vector_correlation_null_comparison"]["observed_gt_null_q95"]

assert time_checks["increments_wrapped_to_pi"]
assert time_checks["null_comparison"]["observed_gt_null_q95"]

final_sanity = {
    "artifact_records": artifact_records,
    "path_root": str(ARTIFACT_BASE.relative_to(BOOK_ROOT)).replace("\\", "/"),
    "all_paths_book_local": all(artifact_is_under(path, ARTIFACT_BASE) for path in artifact_paths),
    "core_identity_checks_passed": True,
    "pdf_used_for": "orientation only",
}
final_path = save_json(final_sanity, TOPIC, "checks", "final-sanity.json")
assert final_path.exists() and final_path.stat().st_size > 100
final_sanity

## Standalone Reading Notes

Use this chapter as a diagnostic sequence rather than as a list of named correlation coefficients. The first decision is always the type of response and predictor. A linear-circular comparison embeds the angle with sine and cosine because an ordinary residual would create a false jump at the cut point. A circular-circular comparison lives naturally on a torus, so the residual must be wrapped after both origins are allowed to move. A spherical-spherical comparison needs rotations and tangent residuals because the fitted value should remain a unit vector rather than drift into Euclidean space.

The permutation panels are part of the lesson. They show what a dependence measure looks like after the relationship has been broken while the marginal directional samples remain intact. That is a safer null comparison than replacing the data with unrelated linear variables. When extending the notebook to a real data set, preserve this pattern: draw the geometry, compute an invariant or equivariant statistic, compare it with a null that respects the sample space, then inspect residuals in angular or tangent coordinates.

The time-series section should be read as a warning against unwrapping by habit. Persistence in directions is visible through lagged angular increments and resultant traces, but the statistic must survive a shift of the angular origin. If a proposed model changes when the compass zero is relabeled, it is measuring the coordinate system rather than the process.


## Takeaways

Directional correlation and regression are coordinate-sensitive until the geometry is made explicit. Embedding an angle, wrapping a residual, fitting a rotation, or shuffling a time order are not implementation details; they define what kind of dependence is being measured.

The recurring pattern is useful beyond this chapter. First choose the sample space, then choose the nuisance transformations that should not matter, then choose a statistic and a visual that expose the same invariant. The final checks enforce that contract: origin shifts, rotations, unit norms, tangent residuals, permutation nulls, and book-local artifacts all agree with the story told by the figures.